# 02 - Model Training Walkthrough

This notebook walks through the same steps as `src/train_model.py`, but interactively,
so you can inspect intermediate results. For the production run, just use:
```bash
python src/train_model.py
```

In [ ]:
import sys
sys.path.append('../src')
import os
os.chdir('..')  # so relative paths like 'data/...' match the project root

from preprocessing import load_data, prepare_train_test_split, build_preprocessor, get_feature_names
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

df = load_data()
X_train, X_test, y_train, y_test = prepare_train_test_split(df)
X_train.shape, X_test.shape

## Train a single baseline model

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', build_preprocessor()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])
pipeline.fit(X_train, y_train)
pipeline.score(X_test, y_test)

## Full comparison across models

For the full multi-model comparison (Logistic Regression, Random Forest, Gradient Boosting)
with saved plots and metrics, run the training script directly - it does everything below
plus saves `models/best_model.pkl` for the API and dashboard to use:

In [ ]:
import subprocess
result = subprocess.run(['python', 'src/train_model.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Inspect saved metrics

In [ ]:
import json
with open('models/metrics.json') as f:
    metrics = json.load(f)
metrics

## Explain a prediction with SHAP
See `explainability.py` and `03_explainability.ipynb`-style usage for more detail.

In [ ]:
from explainability import load_artifacts, explain_instance_shap

pipeline, feature_names, background_sample = load_artifacts()
sample_customer = X_test.iloc[[0]]
explain_instance_shap(pipeline, feature_names, background_sample, sample_customer)